# 🔬 NamML 1 – Phase 2: MinHashLSH Baseline Pipeline
## Dataset: `common-pile/peS2o_filtered`

**Nguồn**: Common Pile v0.1 — phiên bản peS2o đã filter, chỉ giữ bài báo open-license, dùng để train Comma v0.1

| Trường | Mô tả |
|--------|-------|
| `id` | Document ID |
| `text` | Nội dung văn bản (title + body, paragraphs cách nhau `\n\n`) |
| `metadata` | Dict chứa license, source, created, v.v. |

**Pipeline:**
```
common-pile/peS2o_filtered → clean → word N-grams → CountVectorizer (binary) → MinHashLSH → Signatures
```

## 📦 0. Cài đặt thư viện

In [ ]:
# Chạy cell này một lần để cài dependencies
!pip install pyspark datasets -q

## ⚙️ 1. Import & Cấu hình

In [ ]:
import re
import os
import time
import numpy as np

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, ArrayType
from pyspark.ml import Pipeline
from pyspark.ml.feature import CountVectorizer, HashingTF, MinHashLSH

# ── Tham số cấu hình ──────────────────────────────────────────────────────────
DATASET_NAME    = "common-pile/peS2o_filtered"   # ← nguồn mới
N_DOCS          = 50_000       # số doc lấy mẫu (None = toàn bộ)
NGRAM_N         = 5             # word N-gram (5-gram theo paper gốc)
NUM_HASH_TABLES = 10            # số band MinHashLSH
NUM_HTF_BUCKETS = 2 ** 14       # 16 384 buckets cho HashingTF
JACCARD_THRESH  = 0.80          # ngưỡng Jaccard similarity near-dup
OUTPUT_DIR      = "./pes2o_output"
MODEL_PATH      = "./pes2o_minhash_model"

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("✅ Config OK")
print(f"   Dataset        : {DATASET_NAME}")
print(f"   N_DOCS         : {N_DOCS:,}")
print(f"   N-gram order   : {NGRAM_N}")
print(f"   numHashTables  : {NUM_HASH_TABLES}")
print(f"   Jaccard thresh : {JACCARD_THRESH}")

## 🚀 2. Khởi tạo SparkSession

In [ ]:
spark = (
    SparkSession.builder
    .appName("NamML1-Phase2-peS2o-MinHashLSH")
    # ── Memory & Task size fixes ───────────────────────────────────────────
    .config("spark.driver.memory",             "8g")    # tăng heap driver
    .config("spark.driver.maxResultSize",       "4g")    # tránh OOM khi collect
    .config("spark.executor.memory",            "4g")
    .config("spark.sql.shuffle.partitions",     "200")   # đủ partition
    # ── Tránh task quá lớn khi createDataFrame từ list Python ─────────────
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")  # dùng Arrow
    .config("spark.default.parallelism",        "32")
    # ── Serializer nhẹ hơn Java serializer ────────────────────────────────
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .config("spark.kryoserializer.buffer.max",  "512m")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"✅ SparkSession khởi động | version: {spark.version}")
print(f"   driver.memory           : {spark.conf.get('spark.driver.memory')}")
print(f"   serializer              : {spark.conf.get('spark.serializer')}")


## 📥 3. Load dữ liệu `common-pile/peS2o_filtered` → Spark DataFrame

Schema Common Pile: `id` | `text` | `metadata` (dict: license, source, created…)


In [ ]:
import gzip, json, requests, os
from tqdm.auto import tqdm

# ── Bước 1: Stream từ HF → lưu ra file JSONL (tránh OOM khi tạo Spark DF) ──
# Ghi thẳng ra disk thay vì giữ list trong RAM rồi createDataFrame
JSONL_PATH = f"{OUTPUT_DIR}/pes2o_raw.jsonl"

def stream_to_jsonl(n_docs, out_path):
    """Stream common-pile/peS2o_filtered → ghi JSONL từng dòng, không giữ RAM."""
    from datasets import load_dataset

    hf_ds = load_dataset(
        "common-pile/peS2o_filtered",
        split="train",
        streaming=True,
    )
    count = 0
    with open(out_path, "w", encoding="utf-8") as fout:
        for i, row in enumerate(tqdm(hf_ds, total=n_docs, desc="Streaming → JSONL")):
            if n_docs and count >= n_docs:
                break
            text = row.get("text", "") or ""
            if len(text.strip()) < 50:
                continue
            meta = row.get("metadata") or {}
            record = {
                "id":      str(row.get("id", f"doc_{i}")),
                "text":    text,
                "source":  str(meta.get("source",  "")),
                "created": str(meta.get("created", "")),
                "license": str(meta.get("license", "")),
            }
            fout.write(json.dumps(record, ensure_ascii=False) + "\n")
            count += 1
    return count

# ── Chạy stream ──────────────────────────────────────────────────────────────
print(f"📥 Đang stream {N_DOCS:,} docs → {JSONL_PATH}")
t0 = time.time()

if os.path.exists(JSONL_PATH):
    # Đếm dòng hiện có để tránh tải lại
    existing = sum(1 for _ in open(JSONL_PATH))
    if existing >= N_DOCS:
        print(f"   File đã có {existing:,} docs → bỏ qua tải lại")
    else:
        print(f"   File cũ chỉ có {existing:,} docs → tải lại...")
        n_written = stream_to_jsonl(N_DOCS, JSONL_PATH)
        print(f"✅ Đã ghi {n_written:,} docs trong {time.time()-t0:.1f}s")
else:
    n_written = stream_to_jsonl(N_DOCS, JSONL_PATH)
    print(f"✅ Đã ghi {n_written:,} docs trong {time.time()-t0:.1f}s")

# ── Bước 2: Spark đọc JSONL từ disk → tự chia partition, KHÔNG OOM ──────────
# spark.read.json() xử lý file theo partition tự nhiên (mỗi partition ~128 MB)
# → không bao giờ tạo task 358 MB như createDataFrame(list)
print(f"\n📂 Spark đọc JSONL từ disk...")
df_raw = (
    spark.read
    .option("encoding", "UTF-8")
    .json(JSONL_PATH)
    # Ép đúng kiểu, tránh Spark infer schema nhầm
    .select(
        F.col("id").cast("string"),
        F.col("text").cast("string"),
        F.col("source").cast("string"),
        F.col("created").cast("string"),
        F.col("license").cast("string"),
    )
    # Chia thành nhiều partition nhỏ để không có task quá lớn
    .repartition(200)
)

# Cache để tránh đọc lại JSONL nhiều lần
df_raw.cache()
n = df_raw.count()

print(f"\n✅ df_raw: {n:,} docs | {df_raw.rdd.getNumPartitions()} partitions")
print(f"   TB size/partition : ~{os.path.getsize(JSONL_PATH) // df_raw.rdd.getNumPartitions() // 1024} KB")
print(f"📊 Độ dài TB : "
      f"{df_raw.select(F.avg(F.size(F.split('text', ' ')))).first()[0]:.0f} từ/doc")

print("\n📄 Ví dụ doc đầu tiên:")
first = df_raw.first()
print(f"   ID      : {first['id']}")
print(f"   source  : {first['source']}")
print(f"   license : {first['license']}")
print(f"   text    : {first['text'][:200]}...")

df_raw.printSchema()
df_raw.select(
    "id", "source", "license",
    F.substring("text", 1, 80).alias("text_preview")
).show(5, truncate=False)


## 🧹 4. Tiền xử lý & Sinh Word N-grams

peS2o `text` = `title` + `\n\n` + paragraphs (s2orc) hoặc abstract (s2ag).  
Ta clean toàn bộ chuỗi rồi sinh word **5-gram**.

In [ ]:
@F.udf(returnType=StringType())
def clean_text(text: str) -> str:
    """Lowercase, giữ chữ/số, chuẩn hóa khoảng trắng."""
    if not text:
        return ""
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

@F.udf(returnType=ArrayType(StringType()))
def word_ngrams(text: str, n: int = NGRAM_N) -> list:
    """Sinh word N-grams từ chuỗi đã clean."""
    if not text:
        return []
    tokens = text.split()
    if len(tokens) < n:
        return [" ".join(tokens)]
    return [" ".join(tokens[i:i + n]) for i in range(len(tokens) - n + 1)]

df_clean = (
    df_raw
    .withColumn("text_clean", clean_text(F.col("text")))
    .withColumn("ngrams",     word_ngrams(F.col("text_clean")))
)

print(f"✅ Tiền xử lý xong | word {NGRAM_N}-gram")
df_clean.select(
    "id", "source",
    F.size("ngrams").alias("num_ngrams"),
    F.col("ngrams")[0].alias("ngram_example")
).show(truncate=False)

## 🔁 5. Pipeline: CountVectorizer → MinHashLSH

> **Lý do dùng `binary=True`:** MinHashLSH ước lượng **Jaccard similarity** giữa các tập (sets), không phải TF.  
> Binary sparse vector = set membership (0/1) → đúng yêu cầu Spark MLlib.

In [ ]:
# ── 5A. CountVectorizer (binary) ────────────────────────────────────────────
# ⚠ Vocab đã hit cap 131,072 → tăng lên 2^20 (1M) để bao phủ đủ 5-grams
# Với 50K docs x ~200 unique 5-grams/doc → ước tính ~5-10M unique 5-grams
# minDF=2 để loại bỏ 5-grams chỉ xuất hiện 1 lần (noise, tiết kiệm memory)
cv = CountVectorizer(
    inputCol="ngrams",
    outputCol="cv_features",
    minDF=2,              # ← tăng từ 1→2: loại singleton 5-grams (giảm ~60% vocab)
    binary=True,          # bắt buộc cho MinHashLSH (Jaccard similarity)
    vocabSize=2 ** 20,    # ← tăng từ 2^17 → 2^20 (1,048,576): tránh vocab bị cắt
)

# ── 5B. MinHashLSH ────────────────────────────────────────────────────────────
# numHashTables=10: cân bằng precision/recall
# Tăng lên 20 nếu muốn recall cao hơn (phát hiện near-dup tốt hơn)
mh = MinHashLSH(
    inputCol="cv_features",
    outputCol="minhash_signatures",
    numHashTables=NUM_HASH_TABLES,
    seed=42,
)

# ── 5C. Pipeline hoàn chỉnh ───────────────────────────────────────────────────
pipeline = Pipeline(stages=[cv, mh])

print("Đang fit pipeline...")
pipeline_model = pipeline.fit(df_clean)

print("Đang transform...")
df_signed = pipeline_model.transform(df_clean)

cv_model = pipeline_model.stages[0]
mh_model = pipeline_model.stages[1]

vocab_size = len(cv_model.vocabulary)
is_capped  = vocab_size >= 2 ** 20

print(f"\n✅ Pipeline hoàn tất")
print(f"   Vocab size (CountVectorizer) : {vocab_size:,}")
print(f"   Vocab hit cap?               : {'⚠ CÓ — cân nhắc tăng vocabSize' if is_capped else '✅ Không'}")
print(f"   Signature length (cố định)   : {NUM_HASH_TABLES}")

# ── Thống kê vocab & sparsity ─────────────────────────────────────────────────
print("\n📊 Thống kê vector sau CountVectorizer:")
df_signed.select(
    F.avg(F.size("ngrams")).alias("avg_ngrams_per_doc"),
    F.min(F.size("ngrams")).alias("min_ngrams"),
    F.max(F.size("ngrams")).alias("max_ngrams"),
).show()

# Sparsity: số 5-gram active / vocab size
from pyspark.ml.linalg import SparseVector as SV
sample_vec = df_signed.select("cv_features").first()[0]
nnz  = sample_vec.numNonzeros()
size = sample_vec.size
print(f"   Sample vector: {nnz} non-zeros / {size:,} dims (sparsity = {1 - nnz/size:.4f})")


In [ ]:
# Xem kết quả
df_signed.select(
    "id", "source",
    F.size("ngrams").alias("num_ngrams"),
    "cv_features",
    "minhash_signatures"
).show(truncate=False)

## ⚡ 6. Phương án thay thế: HashingTF → MinHashLSH

HashingTF không cần fit vocabulary → phù hợp cho **corpus rất lớn** (toàn bộ peS2o ~38M docs).

In [ ]:
htf = HashingTF(
    inputCol="ngrams",
    outputCol="htf_features",
    numFeatures=NUM_HTF_BUCKETS,
    binary=True,
)
mh_htf = MinHashLSH(
    inputCol="htf_features",
    outputCol="htf_signatures",
    numHashTables=NUM_HASH_TABLES,
    seed=42,
)
pipeline_htf_model = Pipeline(stages=[htf, mh_htf]).fit(df_clean)
df_htf_signed = pipeline_htf_model.transform(df_clean)

print("✅ HashingTF path hoàn tất")
df_htf_signed.select("id", "htf_features", "htf_signatures").show(truncate=False)

## ✅ 7. Kiểm tra độ dài cố định của chữ ký

In [ ]:
sig_sample = df_signed.select("minhash_signatures").first()[0]

print(f"numHashTables (= signature length)  : {len(sig_sample)}")
print(f"Kiểu mỗi phần tử                   : {type(sig_sample[0])}")
print(f"Giá trị hash[0]                     : {sig_sample[0]}")

print("\nKiểm tra tất cả docs có cùng signature length:")
df_signed.select(
    F.size("minhash_signatures").alias("sig_len")
).distinct().show()

## 🔢 8. Tạo cột `flat_signature` (Array[Double]) cho downstream

In [ ]:
# minhash_signatures = Array[VectorUDT]
# Mỗi VectorUDT là DenseVector([hash_value]) → cần UDF để extract
# F.transform() không hoạt động với VectorUDT → dùng UDF

from pyspark.sql.types import ArrayType, DoubleType
from pyspark.ml.linalg import Vector

@F.udf(returnType=ArrayType(DoubleType()))
def extract_flat_signature(signatures):
    """Array[VectorUDT] → Array[Double]: lấy giá trị hash từ mỗi Vector."""
    if signatures is None:
        return None
    return [float(vec[0]) for vec in signatures]

df_final = df_signed.withColumn(
    'flat_signature',
    extract_flat_signature(F.col('minhash_signatures'))
)

print('✅ flat_signature OK')
df_final.select('id', 'flat_signature').show(5, truncate=False)

# Verify độ dài cố định
df_final.select(F.size('flat_signature').alias('sig_len')).distinct().show()


## 🔍 9. Near-Duplicate Detection & Deduplication

Luồng dedup hoàn chỉnh:
```
df_signed
  → approxSimilarityJoin()   # tìm mọi cặp (A, B) có Jaccard ≥ threshold
  → xây đồ thị connected components (Union-Find)
  → mỗi component giữ 1 doc đại diện (doc có text dài nhất)
  → df_deduped                # DataFrame sạch, không còn near-dup
```


In [ ]:
# ════════════════════════════════════════════════════════════════
# BƯỚC 1: Tìm tất cả cặp near-duplicate bằng MinHashLSH
# ════════════════════════════════════════════════════════════════
print(f'Tìm cặp near-dup với Jaccard similarity ≥ {JACCARD_THRESH}...')
t0 = time.time()

# approxSimilarityJoin trả về (datasetA, datasetB, distCol)
# distCol = Jaccard DISTANCE = 1 - similarity
dup_pairs_raw = mh_model.approxSimilarityJoin(
    df_signed, df_signed,
    threshold=1.0 - JACCARD_THRESH,   # distance threshold
    distCol='jaccard_distance',
)

# Chỉ giữ cặp (A != B) và chuẩn hóa thứ tự (A < B) để tránh đếm trùng
dup_pairs = (
    dup_pairs_raw
    .filter(F.col('datasetA.id') != F.col('datasetB.id'))
    .select(
        F.least(   F.col('datasetA.id'), F.col('datasetB.id')).alias('id_a'),
        F.greatest(F.col('datasetA.id'), F.col('datasetB.id')).alias('id_b'),
        F.round(1.0 - F.col('jaccard_distance'), 4).alias('jaccard_sim'),
        F.round(F.col('jaccard_distance'),        4).alias('jaccard_dist'),
    )
    .distinct()   # bỏ cặp (A,B) và (B,A) bị trùng
    .orderBy(F.col('jaccard_sim').desc())
    .cache()
)

n_pairs = dup_pairs.count()
print(f'✅ Tìm được {n_pairs:,} cặp near-dup trong {time.time()-t0:.1f}s')
print()
dup_pairs.show(20, truncate=False)

# Phân bố similarity
print('\n📊 Phân bố Jaccard similarity của các cặp near-dup:')
dup_pairs.select(
    F.round('jaccard_sim', 1).alias('sim_bucket')
).groupBy('sim_bucket').count().orderBy('sim_bucket').show()


In [ ]:
# ════════════════════════════════════════════════════════════════
# BƯỚC 2: Union-Find (Connected Components) trên các cặp near-dup
# ════════════════════════════════════════════════════════════════
# Mỗi nhóm tài liệu near-dup = 1 connected component
# → giữ đúng 1 doc đại diện mỗi nhóm
import pandas as pd

print('Xây dựng connected components (Union-Find)...')
t0 = time.time()

# Lấy tất cả ID docs có trong ít nhất 1 cặp near-dup
pairs_pd = dup_pairs.select('id_a', 'id_b').toPandas()

# ── Union-Find ───────────────────────────────────────────────────────────────
parent = {}

def find(x):
    """Path-compressed find."""
    while parent.get(x, x) != x:
        parent[x] = parent.get(parent.get(x, x), parent.get(x, x))  # path compress
        x = parent.get(x, x)
    return x

def union(x, y):
    px, py = find(x), find(y)
    if px != py:
        parent[px] = py

# Khởi tạo: mỗi ID là root của chính nó
all_ids = set(pairs_pd['id_a']) | set(pairs_pd['id_b'])
for doc_id in all_ids:
    parent[doc_id] = doc_id

# Union tất cả cặp near-dup
for _, row in pairs_pd.iterrows():
    union(row['id_a'], row['id_b'])

# Map mỗi ID → root (component representative)
id_to_root = {doc_id: find(doc_id) for doc_id in all_ids}
component_map = spark.createDataFrame(
    pd.DataFrame(list(id_to_root.items()), columns=['id', 'component_root']),
    schema='id string, component_root string'
)

n_components = len(set(id_to_root.values()))
n_dup_docs   = len(id_to_root)
print(f'✅ Union-Find hoàn tất trong {time.time()-t0:.1f}s')
print(f'   Tổng docs dính near-dup  : {n_dup_docs:,}')
print(f'   Số connected components  : {n_components:,}')
print(f'   Docs cần loại bỏ (dup)   : {n_dup_docs - n_components:,}')


In [ ]:
# ════════════════════════════════════════════════════════════════
# BƯỚC 3: Tạo df_deduped — loại bỏ near-duplicates
# ════════════════════════════════════════════════════════════════
# Chiến lược: mỗi component giữ doc có text DÀI NHẤT
# (thường là bản đầy đủ nhất, không phải bản rút gọn)
print('Loại bỏ near-duplicates...')
t0 = time.time()

# Thêm cột component_root vào df_signed
df_with_component = df_signed.join(
    component_map, on='id', how='left'
).withColumn(
    # Doc không có trong component_map → không có near-dup → root = chính nó
    'component_root', F.coalesce(F.col('component_root'), F.col('id'))
)

# Trong mỗi component: chọn doc có text dài nhất làm đại diện
from pyspark.sql.window import Window

w = Window.partitionBy('component_root').orderBy(
    F.length('text_clean').desc(),  # dài nhất trước
    F.col('id').asc()               # tie-break bằng id
)

df_deduped = (
    df_with_component
    .withColumn('rank', F.rank().over(w))
    .filter(F.col('rank') == 1)          # chỉ giữ rank 1 = đại diện
    .drop('rank', 'component_root')
    .cache()
)

n_before = df_signed.count()
n_after  = df_deduped.count()
n_removed = n_before - n_after

print(f'\n✅ Deduplication hoàn tất trong {time.time()-t0:.1f}s')
print(f'   Docs trước dedup : {n_before:,}')
print(f'   Docs sau dedup   : {n_after:,}')
print(f'   Docs bị loại     : {n_removed:,} ({n_removed/n_before*100:.2f}%)')
print(f'\nSample df_deduped:')
df_deduped.select(
    'id', 'source',
    F.size('ngrams').alias('n_ngrams'),
    F.length('text_clean').alias('text_len'),
).show(10, truncate=False)


In [ ]:
# ════════════════════════════════════════════════════════════════
# BƯỚC 4: Kiểm tra kết quả dedup bằng show_example()
# ════════════════════════════════════════════════════════════════
import pandas as pd

# Lấy 1 component có nhiều dup nhất để kiểm tra
if not pairs_pd.empty:
    top_root = (
        pd.Series(list(id_to_root.values()))
        .value_counts()
        .idxmax()
    )
    members = [k for k, v in id_to_root.items() if v == top_root]
    print(f'Component lớn nhất: root={top_root}, số members={len(members)}')
    print(f'Members: {members}\n')

    # Tạo mini tuning_docs từ các members này để show_example
    df_component = (
        df_signed
        .filter(F.col('id').isin(members))
        .select('id', 'source', 'text_clean')
        .toPandas()
        .rename(columns={'id': 'doc_id', 'source': 'variant_family', 'text_clean': 'text'})
    )
    df_component['source_doc_id'] = top_root
    df_component['variant_type']  = df_component['variant_family']
    df_component['n_words']       = df_component['text'].str.split().str.len()

    print('Nội dung các docs trong component (sẽ bị dedup giữ lại 1):')
    show_example(df_component, top_root)

    # Kiểm tra doc được giữ lại
    kept = df_deduped.filter(F.col('id').isin(members)).select('id', F.length('text_clean').alias('text_len')).toPandas()
    print(f'✅ Doc được giữ lại sau dedup:')
    print(kept)
else:
    print('Không có near-dup pairs → toàn bộ docs là unique.')
    print(f'Thử giảm JACCARD_THRESH (hiện tại={JACCARD_THRESH}) để bắt nhiều cặp hơn.')


In [ ]:
# ════════════════════════════════════════════════════════════════
# BƯỚC 5: Lưu df_deduped ra Parquet
# ════════════════════════════════════════════════════════════════
DEDUPED_PATH = f'{OUTPUT_DIR}/pes2o_deduped.parquet'
DUP_PAIRS_PATH = f'{OUTPUT_DIR}/dup_pairs.parquet'

(
    df_deduped
    .select('id', 'source', 'created', 'license', 'text_clean', 'ngrams', 'flat_signature')
    .write.mode('overwrite')
    .parquet(DEDUPED_PATH)
)
print(f'✅ df_deduped lưu tại: {DEDUPED_PATH}')

dup_pairs.write.mode('overwrite').parquet(DUP_PAIRS_PATH)
print(f'✅ dup_pairs  lưu tại: {DUP_PAIRS_PATH}')

print(f'\n📊 Tóm tắt deduplication:')
print(f'   Input docs  : {n_before:,}')
print(f'   Dup pairs   : {n_pairs:,}')
print(f'   Dup docs    : {n_dup_docs:,} → {n_components:,} components')
print(f'   Output docs : {n_after:,} ({n_removed/n_before*100:.2f}% bị loại)')


## 🎯 10. approxNearestNeighbors – tìm K doc gần nhất

In [ ]:
K = 3
query_row = df_signed.orderBy("id").first()
query_vec = query_row["cv_features"]
query_id  = query_row["id"]

print(f"Query doc id = {query_id}")
print(f"Tìm {K} tài liệu gần nhất theo Jaccard distance:")

(
    mh_model
    .approxNearestNeighbors(df_signed, query_vec, numNearestNeighbors=K)
    .select("id", "source", F.round("distCol", 4).alias("jaccard_dist"))
    .show(truncate=False)
)

## 🔎 13. Kiểm tra trực quan – `show_example()`

Hiển thị tất cả **variants** của một `source_doc_id` từ `tuning_docs`.

| Cột | Ý nghĩa |
|-----|---------|
| `doc_id` | ID document cụ thể |
| `source_doc_id` | ID document gốc (các variant cùng nhóm) |
| `variant_family` | Nhóm biến thể (paraphrase / truncation…) |
| `variant_type` | Loại biến thể cụ thể |
| `n_words` | Số từ |
| `text` | Nội dung văn bản |

In [ ]:
# ── Chuyển df_final (Spark) → pandas tuning_docs ─────────────────────────────
# tuning_docs cần có các cột:
#   doc_id | source_doc_id | variant_family | variant_type | n_words | text
#
# Nếu bạn đã có file riêng, thay bằng:
#   tuning_docs = pd.read_parquet("path/to/tuning_docs.parquet")
#   tuning_docs = pd.read_csv("path/to/tuning_docs.csv")

import pandas as pd

try:
    # ── Load từ file có sẵn ──────────────────────────────────────────────────
    tuning_docs = pd.read_parquet(f"{OUTPUT_DIR}/tuning_docs.parquet")
    print(f"✅ Load tuning_docs từ Parquet: {len(tuning_docs):,} rows")

except FileNotFoundError:
    # ── Tạo tuning_docs từ df_final (Spark → pandas) ─────────────────────────
    print("Chưa có file tuning_docs → tạo từ df_final...")

    # Chuyển Spark DataFrame sang pandas
    pdf = df_final.select(
        "id", "source", "text_clean", "ngrams"
    ).toPandas()

    # Mapping sang schema tuning_docs
    # source_doc_id = id gốc (ở đây mỗi doc là unique, dùng chính id làm gốc)
    tuning_docs = pdf.rename(columns={
        "id":         "doc_id",
        "source":     "variant_family",
        "text_clean": "text",
    }).copy()

    tuning_docs["source_doc_id"]  = tuning_docs["doc_id"]          # 1-to-1 ở data gốc
    tuning_docs["variant_type"]   = tuning_docs["variant_family"]   # s2orc / s2ag
    tuning_docs["n_words"]        = tuning_docs["text"].str.split().str.len()

    # Giữ thứ tự cột chuẩn
    tuning_docs = tuning_docs[
        ["doc_id", "source_doc_id", "variant_family", "variant_type", "n_words", "text"]
    ]

    # Lưu để dùng lần sau
    tuning_docs.to_parquet(f"{OUTPUT_DIR}/tuning_docs.parquet", index=False)
    print(f"✅ Tạo & lưu tuning_docs: {len(tuning_docs):,} rows")

print(f"\nSchema tuning_docs:")
print(tuning_docs.dtypes)
print(f"\nMẫu:")
tuning_docs.head(3)

In [ ]:
def show_example(df, source_doc_id, max_chars: int = 1000):
    """
    Hiển thị tất cả variants của một source_doc_id.

    Parameters
    ----------
    df           : pandas DataFrame với schema tuning_docs
    source_doc_id: ID document gốc cần xem
    max_chars    : số ký tự tối đa hiển thị của text (mặc định 1000)
    """
    sub = df[df["source_doc_id"] == str(source_doc_id)].copy()

    if sub.empty:
        print(f"⚠ Không tìm thấy source_doc_id = {source_doc_id}")
        return

    print(f"\n{'═' * 80}")
    print(f"  SOURCE DOC ID : {source_doc_id}")
    print(f"  Số variants   : {len(sub)}")
    print(f"{'═' * 80}")

    for i, (_, row) in enumerate(sub.iterrows(), 1):
        print(f"\n{'─' * 80}")
        print(f"  Variant {i}/{len(sub)}")
        print(f"{'─' * 80}")
        print(f"  doc_id         : {row['doc_id']}")
        print(f"  variant_family : {row['variant_family']}")
        print(f"  variant_type   : {row['variant_type']}")
        print(f"  n_words        : {row['n_words']}")
        print()
        text_preview = row["text"][:max_chars]
        if len(row["text"]) > max_chars:
            text_preview += f"\n  ... [{len(row['text']) - max_chars:,} ký tự còn lại]"
        print(text_preview)

    print(f"\n{'═' * 80}\n")


# ── Chạy thử với document đầu tiên ───────────────────────────────────────────
example_source_id = tuning_docs["source_doc_id"].iloc[0]
print(f"Chạy show_example với source_doc_id = {example_source_id}")
show_example(tuning_docs, example_source_id)

In [ ]:
# ── Tìm near-duplicate pairs rồi dùng show_example kiểm tra trực quan ────────
# Xem cặp near-dup đầu tiên từ kết quả MinHashLSH

dup_pdf = dup_pairs.toPandas()

if not dup_pdf.empty:
    # Lấy cặp near-dup gần nhất
    top_pair = dup_pdf.iloc[0]
    doc_a    = top_pair["doc_A"]
    doc_b    = top_pair["doc_B"]
    sim      = top_pair["jaccard_sim"]

    print(f"Cặp near-duplicate gần nhất:")
    print(f"  doc_A = {doc_a}")
    print(f"  doc_B = {doc_b}")
    print(f"  Jaccard similarity = {sim:.4f}\n")

    # Hiển thị từng doc trong cặp
    show_example(tuning_docs, doc_a)
    show_example(tuning_docs, doc_b)
else:
    print("Không tìm thấy near-duplicate pairs với ngưỡng hiện tại.")
    print(f"Thử giảm JACCARD_THRESH (hiện tại = {JACCARD_THRESH}).")
    # Fallback: hiển thị 3 doc đầu tiên
    for sid in tuning_docs["source_doc_id"].unique()[:3]:
        show_example(tuning_docs, sid)

## 💾 11. Lưu kết quả & Model

In [ ]:
# Lưu DataFrame có signatures ra Parquet
(
    df_final
    .select("id", "added", "created", "source", "ngrams", "cv_features", "flat_signature")
    .write.mode("overwrite")
    .parquet(f"{OUTPUT_DIR}/df_with_signatures.parquet")
)
print(f"✅ DataFrame lưu tại: {OUTPUT_DIR}/df_with_signatures.parquet")

# Lưu near-duplicate pairs
dup_pairs.write.mode("overwrite").parquet(f"{OUTPUT_DIR}/near_dup_pairs.parquet")
print(f"✅ Near-dup pairs lưu tại: {OUTPUT_DIR}/near_dup_pairs.parquet")

# Lưu pipeline model
pipeline_model.write().overwrite().save(MODEL_PATH)
print(f"✅ Pipeline model lưu tại: {MODEL_PATH}")

## 📊 12. Tóm tắt kết quả

In [ ]:
total_docs = df_final.count()
total_dups = dup_pairs.count()

print("=" * 60)
print("  ✅ Phase 2 – MinHashLSH Pipeline | KẾT QUẢ")
print("=" * 60)
print(f"  Dataset          : {DATASET_NAME} ({DATASET_VERSION})")
print(f"  Tổng documents   : {total_docs:,}")
print(f"  N-gram order     : {NGRAM_N}")
print(f"  Vectorizer       : CountVectorizer (binary=True)")
print(f"  Vocab size       : {len(cv_model.vocabulary):,}")
print(f"  numHashTables    : {NUM_HASH_TABLES}")
print(f"  Signature length : {NUM_HASH_TABLES} (cố định, mọi doc)")
print(f"  Jaccard thresh   : ≥ {JACCARD_THRESH}")
print(f"  Near-dup pairs   : {total_dups:,}")
print(f"  Output dir       : {OUTPUT_DIR}/")
print("=" * 60)

spark.stop()
print("\nSparkSession đã dừng.")